# M1 · Your own physical baseline on the W7900

**Learning objective:** produce a real Qwen-Image-2.1 B0 (BF16, 40 steps, 1024²) result on **your** GPU with a declared protocol, and treat the published 55.206 s median as a *hypothesis to investigate*, never as your answer.

**Predict first:** write down your expected warm latency before running (a W7900 reference is 55.2 s; your cloud instance may differ — that difference is course material, not an error).

Reference lesson: [M01 lesson](../course/M01_baseline/LESSON.md) · [中文](./01_baseline_and_uncertainty.zh-CN.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse explicitly selected physical runs (ids shown + classified)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="01")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

from lab import student_provenance as prov

def _origin_stamp(category: str) -> str:
    """Stamp text for a provenance category; reuse is NEVER labelled as
    measured-today (fail-closed student provenance contract)."""
    return prov.stamp_text(category)

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse earlier physical runs to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing explicitly selected physical run(s) "
          "instead of launching new ones (metered-GPU mode; provenance shown)")
    return False

## 1 · The frozen reference (always available, never yours)
Recompute the published median from the two pinned B0 sessions — this is the number your own run will be compared against.

In [ ]:
from lab import notebook_support as ns
import statistics

raw = ns.reference_warm("B0")
claimed = ns.summary_rows()["B0"]
observed = statistics.median(raw)
assert len(raw) == claimed["n_measure"] == 18
assert abs(observed - claimed["warm_total_median_s"]) < 0.03
_stamp(HISTORICAL, f"B0 reference median {observed:.3f}s "
      f"(min {min(raw):.2f}, max {max(raw):.2f}, n={len(raw)}, 2 sessions x 9)")

## 2 · Run B0 on YOUR W7900
`GPU_CLASSROOM` runs a bounded exercise (1 prompt × cold+warmup+2 measured ≈ 5 min). `GPU_RESEARCH` runs the full frozen protocol (3 prompts × 3 seeds × warmup+3 measured + fresh-prompt arm). Both write append-only records under `runs/<run-id>/`.

In [ ]:
def run_own_b0():
    profile = "classroom" if session.mode == "GPU_CLASSROOM" else "research"
    bundle = session.run("B0", profile=profile)
    return bundle

b0 = gpu_cell(run_own_b0) if want_own_arms() else None
b0_origin = "CURRENT_SESSION_PHYSICAL"
if b0 is None:
    shared = session.latest_campaign_run("B0")
    if shared:
        b0 = session.consume(shared, why="explicit reuse of an existing B0 run "
                                        "(shown run_id; not a new measurement)")
        b0_origin = session.consumed[-1].get("category", "COMMITTED_PHYSICAL_REFERENCE")

## 3 · Your numbers, your statistics
Median, spread, per-step time, and peak VRAM — computed from the records of the run above (or, in REFERENCE mode, still from the frozen campaign, clearly labelled).

In [ ]:
if b0 is None:
    vals, origin, steps_hint = raw, HISTORICAL, 40
    cold = None
else:
    st = b0.stats()
    vals = b0.warm_totals()
    origin = _origin_stamp(b0_origin)
    steps_hint = st["steps"]
    cold = b0.cold_total()

med = statistics.median(vals)
print(f"population n={len(vals)}  median={med:.3f}s  mean={statistics.mean(vals):.3f}s  "
      f"min={min(vals):.3f}s  max={max(vals):.3f}s")
if cold is not None:
    print(f"cold (first load) = {cold:.3f}s  -> warm/cold gap {cold-med:.3f}s")
_stamp(origin, f"n={len(vals)} median={med:.3f}s steps={steps_hint}")
if b0 is not None:
    print("per-step (steady median):", b0.step_stats())
    print("device VRAM peak (MB):", st["vram_device_peak_mb"], "| torch peak:",
          st["vram_peak_mb"])

## 4 · Compare against the published number — explain, don't copy
Reference deviation = your median − 55.206 s. A few percent is normal instance-to-instance variation; a large gap means a different population (cold mixed in), contention (another GPU tenant), or thermal/clock state — name the most likely cause.

In [ ]:
dev = med - 55.206
print(f"reference deviation: {dev:+.3f}s ({dev/55.206*100:+.1f}% vs frozen campaign)")
print("limitation: classroom n is small; treat the deviation as a signal to "
      "investigate, not a verdict")
try:
    fig = rt.plot_ab({"label": "B0: reference vs yours",
                      "classification": "REFERENCE vs " + ("YOUR PHYSICAL RUN"
                      if b0 is not None else "REFERENCE"),
                      "baseline": {"experiment": "B0 ref", "n_warm": 18,
                                   "median_s": 55.206},
                      "candidate": {"experiment": "B0 yours", "n_warm": len(vals),
                                    "median_s": round(med, 3)}})
    if fig: fig.show()
except Exception as e:
    print("plot skipped:", e)

## 5 · Your turn
1. Re-run the bounded exercise and compare the two medians — is the run-to-run spread smaller or larger than the reference deviation?
2. Which single change would most reduce your measurement uncertainty, and why?

In [ ]:
summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## Acceptance criteria

- [ ] You can state which numbers are **measured on this GPU today**, which are
      **historical reference values**, and which are **estimates/simulations**.
- [ ] Every physical run you produced has a visible `run_id` under `runs/`.
- [ ] You did not present a smoke/classroom population as a formal result.
- [ ] You can explain one limitation of each measurement you reported.

## Reproduce & contribute

```bash
python -m lab doctor                                  # admission evidence -> runs/doctor.json
python -m lab run --experiment B0 --date <your-tag>  # full research protocol
python -m lab report --scope student --kind all       # aggregate YOUR runs only
```

Commit your `runs/<run-id>/*.json` records (never `evidence/`) and open a PR
describing: hypothesis → protocol → numbers → limitations. Negative results
are welcome — a measured regression is a scientific outcome, not a failure.